## I. Programas esenciales
1. huggingface_hub (para poder descargar modelos de código abierto)
2. ChromaDB (BBDD en la que vamos a guardar los datos vectorizados)

In [ ]:
!pip install -U transformers accelerate bitsandbytes huggingface_hub

In [ ]:
!pip install chromadb sentence-transformers pymupdf

## II. importacion de librerias
1.   Librerias esenciales
2.   Llamar Token para descargar el modelo de huggin_hub


In [ ]:
import torch, json, time, os
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from huggingface_hub import login
from google.colab import userdata, drive
from sentence_transformers import SentenceTransformer
import chromadb, fitz, os

hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)

- Cargamos y comprobamos rutas donde tendremos nuestros pdfs (Google Drive)
    - Ruta de los archivos vectorizados ( Para persistencia de datos )
    - Ruta de los resultados

In [ ]:
drive.mount("/content/drive")

DRIVE_BASE  = "/content/drive/MyDrive/rag_rioja"
PDF_FOLDER  = f"{DRIVE_BASE}/pdfs"
VECTOR_DB   = f"{DRIVE_BASE}/vectordb_rioja"
RESULTADOS  = f"{DRIVE_BASE}/resultados"
JSON_FOLDER = f"{DRIVE_BASE}/jsons"
RESPUESTAS = f"{DRIVE_BASE}/respuestas"

os.makedirs(RESPUESTAS, exist_ok=True)
os.makedirs(JSON_FOLDER, exist_ok=True)
os.makedirs(PDF_FOLDER, exist_ok=True)
os.makedirs(RESULTADOS, exist_ok=True)

print(f"📁 PDFs en:    {PDF_FOLDER}")
print(f"📁 ChromaDB:   {VECTOR_DB}")
print(f"PDFs encontrados: {os.listdir(PDF_FOLDER)}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📁 PDFs en:    /content/drive/MyDrive/rag_rioja/pdfs
📁 ChromaDB:   /content/drive/MyDrive/rag_rioja/vectordb_rioja
PDFs encontrados: ['boletin_25.pdf', 'fito_15.pdf']


## III. Configuración y Descarga del modelo
1. Configuración BNB(*BitsAndBytesConfig*):
    - carga el modelo en 4bits en vez de 32bits.
    - cuantización NormalFloat4.
2. Aplicar configuración al modelo

- ¿Por qué 4bits ? Se configura de esta manera para consumir de 9-16 GB de VRAM que usaria Gemma-3-4B, de esta manera bajaria a 2.5-3.5 GB de VRAM.
- ¿Empeora la calidad? La diferencia o degradación respecto al 32bits en precisión y retravial es casi minima.

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16
)

model_id = "google/gemma-3-4b-it"
# instanciamos el modelo y le aplicamos la configuración
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

In [ ]:
# cargamos el modelo de embedding y nos conectamos a chromadb
embedding_model = SentenceTransformer("BAAI/bge-m3")
cliente_db      = chromadb.PersistentClient(path=VECTOR_DB)
coleccion       = cliente_db.get_or_create_collection("normativa")

print(f"✅ Colección lista — {coleccion.count()} chunks")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

✅ Colección lista — 0 chunks


##  IV. vectorización de pdfs

*   Cargamos pdf_limpio y vectorizado
*   Elemento de lista



In [ ]:
jsons_en_db  = set(
    m["fuente"] for m in coleccion.get()["metadatas"]
) if coleccion.count() > 0 else set()

jsons_disco  = [f for f in os.listdir(JSON_FOLDER) if f.endswith(".json")]
jsons_nuevos = [f for f in jsons_disco if
                # compara por nombre de PDF origen, no por nombre del JSON
                json.load(open(f"{JSON_FOLDER}/{f}"))["archivo_origen"] not in jsons_en_db]
if jsons_nuevos:
    print(f"⚙️  Indexando {len(jsons_nuevos)} JSON(s) nuevos en ChromaDB...")
    for nombre_json in jsons_nuevos:
        with open(f"{JSON_FOLDER}/{nombre_json}", "r", encoding="utf-8") as f:
            datos = json.load(f)

        textos   = [c["texto"]  for c in datos["chunks"]]
        paginas  = [c["pagina"] for c in datos["chunks"]]
        vectores = embedding_model.encode(
            textos, show_progress_bar=True, batch_size=32
        ).tolist()

        coleccion.add(
            documents = textos,
            embeddings= vectores,
            metadatas = [{"fuente": datos["archivo_origen"], "pagina": p}
                         for p in paginas],
            ids       = [f"{datos['archivo_origen']}_chunk_{i}"
                         for i in range(len(textos))]
        )
        print(f"  ✅ {nombre_json} → {len(textos)} chunks indexados")
else:
    print(f"✅ ChromaDB al día — {coleccion.count()} chunks, nada nuevo")


⚙️  Indexando 2 JSON(s) nuevos en ChromaDB...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

  ✅ boletin_25_chunks.json → 12 chunks indexados


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

  ✅ fito_15_chunks.json → 15 chunks indexados


## V. conexión con chromeDB

In [ ]:
# 3. Función de búsqueda
def buscar_contexto(pregunta, n_resultados=4):
    vector = embedding_model.encode([pregunta]).tolist()
    resultados = coleccion.query(query_embeddings=vector, n_results=n_resultados)

    fuentes = set(m['fuente'] for m in resultados['metadatas'][0])
    print(f"   📎 Fuentes: {fuentes}")

    return "\n\n---\n\n".join(resultados['documents'][0])

## VI. Interfaz gráfica con radio

In [ ]:
!pip install gradio -q

In [ ]:
IN_COLAB = "google.colab" in str(get_ipython())

import gradio as gr
import datetime
# ════════════════════════════════════════════════════════════
#  FUNCIÓN PRINCIPAL — usa model, tokenizer, coleccion,
#  embedding_model definidos en las celdas anteriores
# ════════════════════════════════════════════════════════════

def chat_fn(mensaje, historial):
    """Recibe pregunta, busca contexto en ChromaDB y genera respuesta con Gemma."""
    if not mensaje.strip():
        return historial, ""
    t0 = time.time()

    # 1. Vectorizar la pregunta con el mismo modelo de embeddings
    vector     = embedding_model.encode([mensaje]).tolist()

    # 2. Buscar los 4 chunks más relevantes en ChromaDB
    resultados = coleccion.query(query_embeddings=vector, n_results=4)
    contexto   = "\n\n---\n\n".join(resultados["documents"][0])
    fuentes    = list(set(m["fuente"] for m in resultados["metadatas"][0]))

    # 3. Construir prompt RAG
    prompt = f"""Eres un asistente experto legal y administrativo, especializado en agricultura.
Usa ÚNICAMENTE el siguiente contexto extraído del boletín oficial para responder de forma clara.
Si la respuesta no se encuentra en el contexto, responde: "No tengo suficiente información en el documento para responder a esto".

Contexto:
{contexto}

Pregunta: {mensaje}"""

    # 4. Formatear con el chat template de Gemma
    messages          = [{"role": "user", "content": prompt}]
    prompt_formateado = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    # 5. Inferencia con Gemma (mismos parámetros que en la celda de prueba)
    inputs  = tokenizer(prompt_formateado, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=500,
        temperature=0.1,
        do_sample=True
    )
    input_length = inputs["input_ids"].shape[-1]
    respuesta    = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)

    latencia = round(time.time() - t0, 2)  # ← segundos totales

    # 6. Citar fuentes
    fuentes_texto = "📎 " + " · ".join(fuentes) if fuentes else ""

    historial = historial + [(mensaje, respuesta)]
    # 7. guardar la respuesta en nuestro drive
    registro = {
    "timestamp"  : time.strftime("%Y-%m-%d %H:%M:%S"),
    "pregunta"   : mensaje,
    "respuesta"  : respuesta,
    "fuentes"    : fuentes,
    "latencia_s": latencia,
    }
    nombre_archivo = f"/content/drive/MyDrive/rag_rioja/respuestas/chat_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S_%f')}.json"
    with open(nombre_archivo, "w", encoding="utf-8") as f:
      json.dump(registro, f, ensure_ascii=False, indent=2)

    return historial, fuentes_texto


def clear_chat():
    return [], ""


# ════════════════════════════════════════════════════════════
#  INTERFAZ GRADIO
# ════════════════════════════════════════════════════════════

with gr.Blocks(
    title="AgroConsulta RAG",
    theme=gr.themes.Soft(
        primary_hue=gr.themes.colors.green,
        secondary_hue=gr.themes.colors.orange,
        font=[gr.themes.GoogleFont("DM Sans"), "sans-serif"],
    ),
    css="""
        .source-box {
            background: #f0f7e6;
            border-left: 4px solid #7a9e3b;
            padding: 10px 14px;
            border-radius: 6px;
            font-size: 0.85em;
            color: #4a6741;
        }
        footer { display: none !important; }
        #col-central { max-width: 820px; margin: 0 auto; }
    """,
) as demo:

    gr.Markdown(
        "## 🍇 AgroConsulta RAG\n"
        "Consulta boletines oficiales de agricultura de La Rioja"
    )

    with gr.Column(elem_id="col-central"):

        chatbot = gr.Chatbot(
            label="",
            height=460,
            bubble_full_width=False,
            show_copy_button=True,
        )

        sources_box = gr.Markdown(elem_classes=["source-box"])

        with gr.Row():
            msg_input = gr.Textbox(
                placeholder="Escribe tu pregunta… (Enter para enviar)",
                label="",
                scale=5,
                lines=2,
                max_lines=5,
                autofocus=True,
            )
            send_btn = gr.Button("Enviar ▶", variant="primary", scale=1)

        clear_btn = gr.Button("🗑️  Nueva conversación", variant="stop", size="sm")

    # ── Eventos ─────────────────────────────────────────────
    send_btn.click(
        fn=chat_fn,
        inputs=[msg_input, chatbot],
        outputs=[chatbot, sources_box],
    ).then(lambda: "", outputs=[msg_input])

    msg_input.submit(
        fn=chat_fn,
        inputs=[msg_input, chatbot],
        outputs=[chatbot, sources_box],
    ).then(lambda: "", outputs=[msg_input])

    clear_btn.click(fn=clear_chat, outputs=[chatbot, sources_box])

demo.launch(
    server_name="0.0.0.0",
    share=IN_COLAB,
    show_error=True,
    quiet=False
)

/tmp/ipykernel_8114/1415413935.py:76: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_8114/1415413935.py:76: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_8114/1415413935.py:104: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(
/tmp/ipykernel_8114/1415413935.py:104: DeprecationWarning: The 'show_copy_button' parameter will be removed in Gradio 6.0. You will need to use 'buttons=["copy"]' instead.
  chatbot = gr.Chatbot(
/tmp/ipykernel

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a33a0fe5f8fed269b5.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## VII. Testeo

In [ ]:
MODELO_ID = "gemma3_4b"

def run_rag_query(pregunta: str, n_results: int = 4) -> tuple[str, float]:
    t0 = time.time()

    vector     = embedding_model.encode([pregunta]).tolist()
    resultados = coleccion.query(query_embeddings=vector, n_results=n_results)
    contexto   = "\n\n---\n\n".join(resultados["documents"][0])

    prompt = f"""Eres un asistente experto en agricultura y sanidad vegetal de La Rioja.
Responde ÚNICAMENTE usando el siguiente contexto extraído del Boletín Oficial de Avisos.
Si la respuesta no está en el contexto, di exactamente: "No tengo suficiente información en el documento para responder."
Sé preciso y conciso.

Contexto:
{contexto}

Pregunta: {pregunta}
Respuesta:"""

    import torch
    messages   = [{"role": "user", "content": prompt}]
    prompt_fmt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs     = tokenizer(prompt_fmt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=300,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    input_len = inputs["input_ids"].shape[-1]
    respuesta = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()
    return respuesta, time.time() - t0

Mounted at /content/drive


In [ ]:

PREGUNTAS = [
    {"id": "Q01", "pregunta": "¿Qué enfermedad fúngica puede causar daños durante el desborre en la variedad Garnacha?"},
    {"id": "Q02", "pregunta": "¿En qué estados fenológicos se recomienda tratar contra la excoriosis?"},
    {"id": "Q03", "pregunta": "¿Cómo se llaman popularmente los Agrotis sp. y en qué momento del día causan daños?"},
    {"id": "Q04", "pregunta": "¿Qué síntomas produce el virus GPGV en las cepas de La Rioja?"},
    {"id": "Q05", "pregunta": "¿A través de qué ácaro se transmite el virus GPGV en campo?"},
    {"id": "Q06", "pregunta": "¿Cuál es la carencia nutricional más frecuente en viñedos riojanos?"},
    {"id": "Q07", "pregunta": "¿Cuántas generaciones tiene la polilla del racimo Lobesia botrana en La Rioja?"},
    {"id": "Q08", "pregunta": "¿Cuál es la superficie mínima para aplicar confusión sexual contra la polilla?"},
    {"id": "Q09", "pregunta": "¿Cuándo se colocan los difusores de feromonas en Rioja Media?"},
    {"id": "Q10", "pregunta": "¿Con qué Real Decreto se relaciona la confusión sexual como gestión integrada?"},
]

raw_results = []
for item in PREGUNTAS:
    print(f"→ {item['id']}...")
    respuesta, latencia = run_rag_query(item["pregunta"])
    raw_results.append({
        "id":               item["id"],
        "respuesta_modelo": respuesta,
        "latencia_s":       round(latencia, 2),
        "modelo":           MODELO_ID,
    })
    print(f"  ✅ {latencia:.1f}s")

output_path = f"{RESULTADOS}/eval_{MODELO_ID}.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(raw_results, f, ensure_ascii=False, indent=2)

print(f"\n✅ {output_path}")

→ Q01...
  ✅ 8.6s
→ Q02...
  ✅ 5.8s
→ Q03...
  ✅ 6.2s
→ Q04...
  ✅ 13.3s
→ Q05...
  ✅ 6.7s
→ Q06...
  ✅ 6.1s
→ Q07...
  ✅ 4.4s
→ Q08...
  ✅ 6.1s
→ Q09...
  ✅ 4.0s
→ Q10...
  ✅ 3.8s

✅ /content/drive/MyDrive/testing/data/results/eval_gemma3_4b.json
